<a href="https://colab.research.google.com/github/EmanHrustemovic/FlyRank-AI-Intership-ML-Track-/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


1. Two paper findings + my methodology questions

Source: FlyRank — "The State of AI-Driven SEO in Numbers" (March 2026), public data report.

Finding A — "The Freshness Multiplier" (Finding #4)

Claim: 365+ day old content that was refreshed within the last 30 days shows a 3.2x health-score boost (10.7 → 34.5) and a 57x impression boost (71 → 4,039). The paper calls this "one of the strongest measured levers available" and its most dramatic finding.

Methodology question: The paper itself flags that the adjacent 361+ freshness bucket in this same finding has an unstable 283:1 growth ratio, driven by just 1 declining page in that bucket — and it explicitly warns readers not to treat that number as a headline multiplier. The 57x impression-boost figure sits right next to that flagged instability, but no sample size (n) is shown for the specific "365+ days old, refreshed in last 30 days" cohort it's drawn from. Given the paper's own demonstrated discipline about flagging small-n instability elsewhere in this exact finding, was the same n-check applied to this headline number, and if the cohort is small, would it carry the same caveat the neighboring bucket receives?

This is asked constructively — the paper already shows real rigor by self-flagging instability once in this section; the question is whether that same check was run consistently on the number chosen as the single most dramatic finding in the report.

Finding B — ML Appendix: "What Predicts Growth?" (logistic regression, 71% holdout accuracy)

Claim: Content Age is the strongest negative predictor of growth, while Days Since Update and Days Visible are among the strongest positive predictors.

Methodology question: Where does the growth/decline label come from relative to the feature window? If a page is labeled "growing" based on rising impressions over a tracked period, and two of the three strongest predictors — Days Since Update and Days Visible — are themselves measures of time/presence within that same tracked period, is there a risk that these "predictors" are partially describing the same window used to construct the label, rather than acting as independent, forward-looking signal? Concretely: would a page that's been "visible" longer simply have had more chances to accumulate the impressions that define it as "growing" in the first place, independent of anything the model is meant to be learning?

I'm asking this exact question about my own model in Section 3 below (leakage audit) — raising it here first, in the same spirit, before turning it on my own work.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

con = duckdb.connect()
hf_token = userdata.get('eman')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

query = """
WITH daily AS (
    SELECT * FROM read_parquet('""" + rel + """/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
),
early AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impr_early,
        SUM(gsc_clicks) AS clicks_early,
        SUM(gsc_sum_position) AS sum_pos_early
    FROM daily WHERE report_date <= DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),
late AS (
    SELECT content_hash_id,
        SUM(gsc_impressions) AS impr_late,
        SUM(gsc_sum_position) AS sum_pos_late
    FROM daily WHERE report_date > DATE '2026-03-15'
    GROUP BY content_hash_id
),
labeled AS (
    SELECT
        e.client_hash_id, e.content_hash_id, e.impr_early, e.clicks_early,
        (e.sum_pos_early * 1.0 / NULLIF(e.impr_early,0)) AS avg_position_early,
        (e.clicks_early * 1.0 / NULLIF(e.impr_early,0)) AS ctr_early,
        CASE WHEN (l.sum_pos_late * 1.0 / NULLIF(l.impr_late,0))
             > (e.sum_pos_early * 1.0 / NULLIF(e.impr_early,0)) THEN 1 ELSE 0 END AS is_declining_label
    FROM early e JOIN late l ON e.content_hash_id = l.content_hash_id
    WHERE e.impr_early > 0 AND l.impr_late > 0
)
SELECT lb.*, dc.word_count, DATE '2026-03-15' - dc.content_updated_date AS days_since_update
FROM labeled lb
LEFT JOIN read_parquet('""" + rel + """/dim_content.parquet') dc ON lb.content_hash_id = dc.content_hash_id
"""

df2 = con.sql(query).df().dropna()
X2 = df2[['impr_early', 'avg_position_early', 'ctr_early', 'word_count', 'days_since_update']]
y2 = df2['is_declining_label']
groups = df2['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X2, y2, groups))
X_train_g, X_test_g = X2.iloc[train_idx], X2.iloc[test_idx]
y_train_g, y_test_g = y2.iloc[train_idx], y2.iloc[test_idx]

print("Client overlap between train/test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])), "(should be 0)")

def precision_at_k(y_true, scores, k=50):
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].mean()

logreg_g = LogisticRegression(max_iter=1000).fit(X_train_g, y_train_g)
scores_g = logreg_g.predict_proba(X_test_g)[:, 1]
p50_grouped = precision_at_k(y_test_g.reset_index(drop=True), pd.Series(scores_g), k=50)
auc_grouped = roc_auc_score(y_test_g, scores_g)

print(f"BEFORE (w05, random split):  Precision@50: 0.960 | AUC: 0.675")
print(f"AFTER  (w06, client-grouped split): Precision@50: {p50_grouped:.3f} | AUC: {auc_grouped:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Client overlap between train/test: 0 (should be 0)
BEFORE (w05, random split):  Precision@50: 0.960 | AUC: 0.675
AFTER  (w06, client-grouped split): Precision@50: 0.960 | AUC: 0.649


Before (w05, random split): Precision@50 = 0.960, AUC = 0.675
After (w06, client-grouped split, 0 client overlap confirmed): Precision@50 = 0.960, AUC = 0.649

Precision@50 held steady under the stricter split, which is reassuring — the top-50
ranked predictions aren't relying on client-specific patterns leaking across train/test.
AUC dropped modestly (0.675 → 0.649), suggesting a small amount of the original AUC was
inflated by the model learning client-level quirks under the random split. The drop is
small enough that I'm treating the original model as reasonably validated, not
compromised — but the grouped split is the more honest number to report going forward.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Leakage audit: confirmed the feature set contains only early-window and static content
data (impr_early, avg_position_early, ctr_early, word_count, days_since_update) — no
late-window or label-derived columns present (explicit check above returns an empty list).

Correlation with is_declining_label: strongest is avg_position_early at -0.255, followed
by days_since_update at -0.072. None are anywhere near 1.0, which would be the signature
of a feature that's effectively the label in disguise. The features carry real but modest
individual signal — consistent with a model that needs to combine them jointly (as
Logistic Regression does) rather than relying on any single leaked shortcut.

In [7]:
print("Features used:", list(X2.columns))
print("Any late-window columns present:", [c for c in X2.columns if 'late' in c.lower()])

corr_check = df2[['impr_early', 'avg_position_early', 'ctr_early',
                   'word_count', 'days_since_update', 'is_declining_label']].corr()['is_declining_label']
print("\nCorrelation with label:")
print(corr_check)

Features used: ['impr_early', 'avg_position_early', 'ctr_early', 'word_count', 'days_since_update']
Any late-window columns present: []

Correlation with label:
impr_early           -0.009093
avg_position_early   -0.255330
ctr_early             0.013385
word_count           -0.047855
days_since_update    -0.071519
is_declining_label    1.000000
Name: is_declining_label, dtype: float64


Claim rewrite: Original — "The model beat the simple rule — Precision@50 went from 0.70
to 0.96, meaning almost every page it flagged as declining actually was."

Rewritten — "On this dataset and this test split, the Logistic Regression model showed
a measured Precision@50 of 0.96, compared to 0.70 for the baseline rule — a directional
improvement that held up under a stricter, client-grouped re-evaluation. This is an
observed result on historical data, not a guarantee of future performance, and should be
treated as decision-support for prioritization, not as a certified accuracy claim."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.